# Cheap vs searched, over 5 eval seeds
`scripts/uc_seeds.sh`'s runs: ESSI / OSP / adaptive (auto-k on accuracy, auto γ, auto α - no image search) vs SAeUron (its τ × γ image search), each block on its own, as `uc_priority_comparison.ipynb` - but on 5 eval seeds. Nothing is searched again: the features, γ and α are the priority runs'; only the evaluation images are new.

- **removal**: the answer set at each of UnlearnCanvas's seeds 188, 288, 588, 688, 888 (1,000 images per seed, as the seed-188 numbers)
- **insertion**: 5 copies of the 20 base images (`seed_rep` 0 = the original bases, j > 0 at seed + 1000 j)

Every metric is computed per seed exactly as in `uc_priority_comparison.ipynb` (means over the 20 objects, EPR pooled over the images, FID per seed), then **mean ± std over the 5 seeds**.

In [ ]:
import os
import numpy as np
import pandas as pd

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)

OUTPUTS = "outputs_sdxlcls_ucbase_seeds5"   # sync evaluation/outputs_sdxlcls_ucbase_seeds5
REMOVAL_PATH = f"{OUTPUTS}/uc_results.csv"
INJECT_PATH = f"{OUTPUTS}/uc_inject_results.csv"
RUN = "evaluation/uc_auto"
TARGET = 0.95  # scripts/uc_seeds.sh's TARGET
BLOCKS = {"down_blocks.2.attentions.1": "down.2.1", "mid_block.attentions.0": "mid.0",
          "up_blocks.0.attentions.0": "up.0.0", "up_blocks.0.attentions.1": "up.0.1"}
AUTOK = {"ESSI": "saeuron_masked", "OSP": "osp", "adaptive": "adaptive"}   # name -> --rules (= the rows' kind)
CHEAP = list(AUTOK)
SEARCHED = ["SAeUron"]


def saeuron_dir(block):
    return f"{RUN}/saeuron_all_searched_toptau_{block.replace('.', '_')}"


def load(path):
    if not os.path.exists(path):
        print(f"!! {path} missing - sync it from the cluster")
        return pd.DataFrame(columns=["out_dir", "subject", "block", "kind", "k_metric", "strength"])
    return pd.read_csv(path)


removal, inject = load(REMOVAL_PATH), load(INJECT_PATH)
for t, col in [(removal, "seed"), (inject, "seed_rep")]:
    if len(t) and col not in t:
        print(f"!! no {col} column - these tables weren't written with --per_seed")
print(f"{len(removal)} removal rows, seeds {sorted(removal['seed'].unique()) if 'seed' in removal else '-'}; "
      f"{len(inject)} insertion rows, seed_rep {sorted(inject['seed_rep'].unique()) if 'seed_rep' in inject else '-'}")

## Settings
As `uc_priority_comparison.ipynb`: removal at auto γ (cheap) / searched γ (SAeUron); insertion at α = −that γ and at auto α.

In [ ]:
def removal_settings(block):
    s = {}
    for name, rule in AUTOK.items():
        d = f"{RUN}/{rule}_all_autok_accuracy_autog"
        s[(name, "auto γ")] = lambda t, d=d, rule=rule: (t["out_dir"] == d) & (t["kind"] == rule) & (t["k_metric"] == "accuracy")
    s[("SAeUron", "searched γ")] = lambda t, d=saeuron_dir(block): (t["out_dir"] == d) & (t["kind"] == "saeuron")
    return s


def inject_settings(block):
    s = {}
    auto = f"_sauto{TARGET:g}"
    for name, rule in AUTOK.items():
        d = f"{RUN}/{rule}_all_autok_accuracy_autog"
        s[(name, "α = −γ")] = lambda t, d=d, rule=rule: ((t["out_dir"] == d) & (t["kind"] == rule)
                                                        & (t["k_metric"] == "accuracy_snegg"))
        s[(name, "auto α")] = lambda t, d=d, rule=rule: ((t["out_dir"] == d) & (t["kind"] == rule)
                                                        & (t["k_metric"] == f"accuracy{auto}"))
    d = saeuron_dir(block)
    s[("SAeUron", "α = −γ")] = lambda t, d=d: ((t["out_dir"] == d) & (t["kind"] == "saeuron")
                                              & t["k_metric"].astype(str).str.endswith("_snegg"))
    s[("SAeUron", "auto α")] = lambda t, d=d: ((t["out_dir"] == d) & (t["kind"] == "saeuron")
                                              & t["k_metric"].astype(str).str.endswith(auto))
    return s


def rows(table, settings, block):
    at = table[table["block"] == block] if len(table) else table
    return {k: at[f(at)] if len(at) else at for k, f in settings(block).items()}


coverage = {}
for block, short in BLOCKS.items():
    for kind, table, settings, col in [("removal", removal, removal_settings, "seed"),
                                       ("insertion", inject, inject_settings, "seed_rep")]:
        for k, t in rows(table, settings, block).items():
            n = t.groupby(col)["subject"].nunique() if len(t) and col in t else pd.Series(dtype=int)
            coverage.setdefault((kind, *k), {})[short] = f"{len(n)} seeds, {n.min() if len(n) else 0}-{n.max() if len(n) else 0} obj"
print("seeds x objects with results (of 5 x 20):")
pd.DataFrame(coverage).T.reindex(columns=list(BLOCKS.values())).fillna("-")

## Per seed, then mean ± std
`removal_row` / `inject_row` are `uc_priority_comparison.ipynb`'s, applied to one seed's rows. `per_seed[(block, side)]` keeps every seed's numbers; `stats[(block, side)]` the mean and std over the seeds.

In [ ]:
UNLEARN = ["UA", "IRA", "CRA", "fid"]
INJECT = {"EPR": "EPR ↑", "uc_classified_as": "classified as ↑", "uc_p_target_gain": "p(object) gain ↑",
          "mask_iou": "mask IoU ↑", "base_subject_remaining": "subject left ↓", "background_psnr": "bg PSNR ↑", "fid": "fid ↓"}


def epr(t):
    if "epr_target_change" not in t:
        return np.nan
    ok = t["epr_target_change"].notna() & t["epr_nontarget_change"].notna()
    if not ok.any():
        return np.nan
    w = t.loc[ok, "n_images"].astype(float)
    return (np.average(t.loc[ok, "epr_target_change"], weights=w)
            / (np.average(t.loc[ok, "epr_nontarget_change"], weights=w) + 1e-8))


def removal_row(t):
    out = {c: t[c].mean() for c in UNLEARN if c in t}
    out["(UA + IRA) / 2"] = (out.get("UA", np.nan) + out.get("IRA", np.nan)) / 2
    out["EPR"] = epr(t)
    out["γ"] = t["gamma"].mean()
    out["objects"] = t["subject"].nunique()
    return out


def inject_row(t):
    w = t["n_images"].astype(float)
    out = {INJECT[c]: np.average(t[c], weights=w) if t[c].notna().all() else t[c].mean()
           for c in INJECT if c in t and c != "EPR" and t[c].notna().any()}
    out["EPR ↑"] = epr(t)
    out["α"] = t["strength"].mean()
    out["objects"] = t["subject"].nunique()
    return out


def by_seed(table, settings, block, col, row_fn):
    """(method, setting, seed) -> that seed's row."""
    out = {}
    for k, t in rows(table, settings, block).items():
        if len(t) and col in t:
            for seed, g in t.groupby(col):
                out[(*k, seed)] = row_fn(g)
    return pd.DataFrame(out).T


def mean_std(t):
    """(method, setting, seed) rows -> (method, setting) x (metric, mean / std / seeds)."""
    if t.empty:
        return t
    t = t.astype(float)
    g = t.groupby(level=[0, 1], sort=False)
    out = pd.concat({"mean": g.mean(), "std": g.std(ddof=1)}, axis=1).swaplevel(axis=1)
    out[("seeds", "n")] = g.size()
    return out


per_seed, stats = {}, {}
for block, short in BLOCKS.items():
    per_seed[(short, "removal")] = by_seed(removal, removal_settings, block, "seed", removal_row)
    per_seed[(short, "insertion")] = by_seed(inject, inject_settings, block, "seed_rep", inject_row)
    for side in ["removal", "insertion"]:
        stats[(short, side)] = mean_std(per_seed[(short, side)])


def fmt(s, cols):
    """mean ± std strings for display."""
    if s.empty:
        return s
    return pd.DataFrame({c: [f"{m:.3f} ± {d:.3f}" if pd.notna(m) else "-"
                             for m, d in zip(s[(c, "mean")], s[(c, "std")])]
                         for c in cols if (c, "mean") in s}, index=s.index).assign(seeds=s[("seeds", "n")])


for short in BLOCKS.values():
    print(f"=== {short}: removal")
    display(fmt(stats[(short, "removal")], ["UA", "IRA", "CRA", "(UA + IRA) / 2", "fid", "EPR", "γ"]))
    print(f"=== {short}: insertion")
    display(fmt(stats[(short, "insertion")], ["EPR ↑", "classified as ↑", "p(object) gain ↑", "fid ↓", "α"]))

## Cheap vs searched
The paper's claim, per block: each cheap method against SAeUron's search on removal (UA + IRA) / 2, removal FID and insertion EPR. `Δ` = cheap − SAeUron per seed (same answer images / base copy for both, so paired), mean ± std over the seeds; `wins` = seeds where the cheap method is better (lower is better for FID).

In [ ]:
CLAIM = [("removal", "(UA + IRA) / 2", True), ("removal", "fid", False),
         ("insertion", "EPR ↑", True)]
SETTING = {"removal": {"cheap": "auto γ", "searched": "searched γ"},
           "insertion": {"cheap": "α = −γ", "searched": "α = −γ"}}

claim = {}
for short in BLOCKS.values():
    for side, col, higher in CLAIM:
        t = per_seed[(short, side)].sort_index()  # sorted: the (method, setting) lookups below
        if t.empty or col not in t:
            continue
        ref_key = (SEARCHED[0], SETTING[side]["searched"])
        if ref_key not in t.index.droplevel(2):
            continue
        ref = t.loc[ref_key, col].astype(float)
        for m in CHEAP:
            if (m, SETTING[side]["cheap"]) not in t.index.droplevel(2):
                continue
            d = (t.loc[(m, SETTING[side]["cheap"]), col].astype(float) - ref).dropna()
            wins = int((d > 0).sum() if higher else (d < 0).sum())
            claim[(short, m, f"{side} {col}")] = {"Δ mean": d.mean(), "Δ std": d.std(ddof=1),
                                                    "wins": f"{wins}/{len(d)}"}
claim = pd.DataFrame(claim).T
claim

## LaTeX export
One table per block, as `uc_priority_comparison.ipynb`'s per-block tables, with mean ± std over the 5 seeds: removal FID and (UA + IRA) / 2 at the method's γ, insertion EPR at α = −γ and at auto α. Best mean per column **bold**. Written to `LAYER_DIR`.

In [ ]:
LAYER_DIR = "layer_tables_seeds5"
TEX_NAMES = {"ESSI": "ESSI", "adaptive": "Adaptive", "OSP": "OSP", "SAeUron": "SAeUron"}
GAMMA = {"ESSI": "auto γ", "OSP": "auto γ", "adaptive": "auto γ", "SAeUron": "searched γ"}
COLS = [("FID \\(\\downarrow\\)", "removal", None, "fid", False),
        ("\\(\\frac{\\text{UA}+\\text{IRA}}{2}\\) \\(\\uparrow\\)", "removal", None, "(UA + IRA) / 2", True),
        ("EPR \\(\\uparrow\\) (\\(\\alpha = -\\gamma\\))", "insertion", "α = −γ", "EPR ↑", True),
        ("EPR \\(\\uparrow\\) (\\(\\alpha_{\\text{auto}}\\))", "insertion", "auto α", "EPR ↑", True)]


def layer_tex(short):
    vals = {}
    for m in TEX_NAMES:
        row = []
        for _, side, setting, col, _ in COLS:
            s = stats[(short, side)]
            k = (m, setting or GAMMA[m])
            ok = not s.empty and k in s.index and (col, "mean") in s
            row.append((s.loc[k, (col, "mean")], s.loc[k, (col, "std")]) if ok else (np.nan, np.nan))
        vals[m] = row
    best = [(max if higher else min)([v[j][0] for v in vals.values() if pd.notna(v[j][0])], default=None)
            for j, (*_, higher) in enumerate(COLS)]

    def cell(j, mean, std):
        if pd.isna(mean):
            return "--"
        text = f"{mean:.3f} \\(\\pm\\) {std:.3f}"
        return rf"\textbf{{{text}}}" if best[j] is not None and round(mean, 3) == round(best[j], 3) else text

    lines = [r"\begin{table}[t]", r"\centering",
             rf"\caption{{Removal FID and \(\frac{{\text{{UA}}+\text{{IRA}}}}{{2}}\) and insertion EPR at \texttt{{{short}}}, "
             r"mean \(\pm\) std over 5 evaluation seeds (each seed: means over the 20 objects, EPR pooled over its "
             r"edited images). ESSI, OSP and Adaptive use \(\gamma_{\text{auto}}\) and no image search; SAeUron its "
             r"searched \(\tau\) and \(\gamma\). Best mean per column in \textbf{bold}.}",
             rf"\label{{table:layer_{short.replace('.', '')}_seeds}}",
             r"\resizebox{\columnwidth}{!}{%", r"\begin{tabular}{|l|" + "c" * len(COLS) + "|}", r"\hline",
             "    Method & " + " & ".join(c[0] for c in COLS) + r" \\", r"\hline"]
    lines += [f"    {TEX_NAMES[m]} & " + " & ".join(cell(j, *v) for j, v in enumerate(row)) + r" \\"
              for m, row in vals.items() if not all(pd.isna(v[0]) for v in row)]
    lines += [r"\hline", r"\end{tabular}}", r"\end{table}"]
    return "\n".join(lines) + "\n"


os.makedirs(LAYER_DIR, exist_ok=True)
for short in BLOCKS.values():
    path = os.path.join(LAYER_DIR, f"layer_{short.replace('.', '')}_seeds5.tex")
    open(path, "w").write(layer_tex(short))
    print(path)